# Qwen-14B Next-Step Map Reasoning Probe

This notebook tests whether `Qwen/Qwen2.5-14B-Instruct` can choose the immediate next navigation command toward a target room or target object.

This is closer to the real LPLH2 loop than asking for a full long route. In the game, the agent gets a fresh LLM call after every move, so it only needs to choose the next step correctly.

The test uses one static discovered Zork1-style map and asks about 20 next-step decisions. It compares four representations:

- **current_kg_format**: current LPLH2 KG prompt shape, `map -> room -> temp_have/have/direction/may_direction`.
- **raw_graph**: clean nested `navigation_graph` JSON.
- **json_edge_list**: current-room JSON plus explicit edge strings like `Living Room --down--> Cellar`.
- **assisted_next_step**: clean JSON plus one code-computed target-specific next-step hint.

The output is graded only on `known` and `next_command`.


In [ ]:
!pip -q install -U transformers accelerate sentencepiece


In [ ]:
import copy, json, re
from collections import deque
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-14B-Instruct"
MAX_NEW_TOKENS = 192

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("CUDA available:", torch.cuda.is_available())


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print("Loaded", MODEL_NAME, "with dtype", dtype)


## Static World Model

The graph is intentionally small but nontrivial. Some targets are several rooms away, but the model only has to return the first command.


In [ ]:
BASE_WORLD_MODEL = {
    "current_location": "Living Room",
    "current_room_state": {
        "inventory": ["sword", "lantern", "sack", "bottle"],
        "visible_objects": ["trophy case", "rug", "trap door"],
        "object_states": {
            "trophy case": "open",
            "rug": "moved",
            "trap door": "revealed, open",
        },
        "confirmed_exits": {"east": "Kitchen", "down": "Cellar"},
        "blocked_exits": {"west": "nailed wooden door"},
        "untried_exits": ["north", "south"],
    },
    "navigation_graph": {
        "West of House": {"north": "North of House"},
        "North of House": {"south": "West of House", "north": "Forest Path", "east": "Behind House"},
        "Forest Path": {"south": "North of House", "north": "Clearing", "up": "Up a Tree"},
        "Up a Tree": {"down": "Forest Path"},
        "Clearing": {"south": "Forest Path"},
        "Behind House": {"west": "North of House", "east": "Kitchen"},
        "Kitchen": {"west": "Living Room", "east": "Behind House", "up": "Attic"},
        "Attic": {"down": "Kitchen"},
        "Living Room": {"east": "Kitchen", "down": "Cellar"},
        "Cellar": {"up": "Living Room", "north": "Troll Room", "south": "East-West Passage"},
        "Troll Room": {"south": "Cellar", "east": "East-West Passage"},
        "East-West Passage": {"west": "Troll Room", "east": "Loud Room", "north": "Round Room"},
        "Loud Room": {"west": "East-West Passage"},
        "Round Room": {"south": "East-West Passage"},
    },
    "known_object_locations": {
        "egg": "Up a Tree",
        "lantern": "Living Room",
        "sword": "Living Room",
        "sack": "Kitchen",
        "bottle": "Kitchen",
        "axe": "Troll Room",
        "leaflet": "West of House",
    },
    "frontier": {
        "Clearing": ["east", "west", "north"],
        "Cellar": ["west", "down"],
        "East-West Passage": ["down"],
        "Loud Room": ["up"],
    },
}

ROOM_OBJECTS = {
    "West of House": ["mailbox", "leaflet"],
    "North of House": ["white house", "path"],
    "Forest Path": ["tree"],
    "Up a Tree": ["bird nest", "egg"],
    "Clearing": ["pile of leaves"],
    "Behind House": ["window"],
    "Kitchen": ["sack", "bottle", "table"],
    "Attic": [],
    "Living Room": ["trophy case", "rug", "trap door"],
    "Cellar": [],
    "Troll Room": ["axe"],
    "East-West Passage": [],
    "Loud Room": [],
    "Round Room": [],
}

ROOM_LOCAL_STATE = {
    "Living Room": {
        "visible_objects": ["trophy case", "rug", "trap door"],
        "object_states": {"trophy case": "open", "rug": "moved", "trap door": "revealed, open"},
        "blocked_exits": {"west": "nailed wooden door"},
        "untried_exits": ["north", "south"],
    },
    "Kitchen": {
        "visible_objects": ["sack", "bottle", "table"],
        "object_states": {},
        "blocked_exits": {},
        "untried_exits": [],
    },
    "Forest Path": {
        "visible_objects": ["tree"],
        "object_states": {},
        "blocked_exits": {},
        "untried_exits": [],
    },
    "Up a Tree": {
        "visible_objects": ["bird nest", "egg"],
        "object_states": {},
        "blocked_exits": {},
        "untried_exits": [],
    },
    "Troll Room": {
        "visible_objects": ["axe"],
        "object_states": {"troll": "defeated or absent"},
        "blocked_exits": {},
        "untried_exits": [],
    },
    "Loud Room": {
        "visible_objects": [],
        "object_states": {"condition": "echo may distort commands"},
        "blocked_exits": {},
        "untried_exits": ["up"],
    },
}

def shortest_route(graph, start, target):
    if start not in graph or target not in graph:
        return None
    q = deque([(start, [], [start])])
    seen = {start}
    while q:
        room, route, path = q.popleft()
        if room == target:
            return {"route": route, "path": path}
        for direction, nxt in graph.get(room, {}).items():
            if nxt not in seen:
                seen.add(nxt)
                q.append((nxt, route + [direction], path + [nxt]))
    return None

def local_state_for(start):
    graph = BASE_WORLD_MODEL["navigation_graph"]
    local = ROOM_LOCAL_STATE.get(start, {"visible_objects": [], "object_states": {}, "blocked_exits": {}, "untried_exits": []})
    return {
        "inventory": BASE_WORLD_MODEL["current_room_state"]["inventory"],
        "visible_objects": local["visible_objects"],
        "object_states": local["object_states"],
        "confirmed_exits": graph.get(start, {}),
        "blocked_exits": local["blocked_exits"],
        "untried_exits": local["untried_exits"],
    }

def build_clean_world(case, include_next_step_hint=False):
    start = case["start"]
    world = {
        "current_location": start,
        "current_room_state": local_state_for(start),
        "navigation_graph": BASE_WORLD_MODEL["navigation_graph"],
        "known_object_locations": BASE_WORLD_MODEL["known_object_locations"],
        "frontier": BASE_WORLD_MODEL["frontier"],
    }
    if include_next_step_hint:
        expected = expected_for_case(case)
        if expected["known"]:
            world["target_next_step_hint"] = {
                "target_location": expected.get("target_location"),
                "target_object": expected.get("target_object"),
                "next_command": expected["next_command"],
                "next_room": expected["next_room"],
            }
    return world

def build_edge_world(case):
    graph = BASE_WORLD_MODEL["navigation_graph"]
    edges = []
    for src, exits in graph.items():
        for direction, dst in exits.items():
            edges.append(f"{src} --{direction}--> {dst}")
    return {
        "current_location": case["start"],
        "current_room_state": local_state_for(case["start"]),
        "known_object_locations": BASE_WORLD_MODEL["known_object_locations"],
        "navigation_edges": edges,
        "frontier": BASE_WORLD_MODEL["frontier"],
    }

def build_current_kg_world(case):
    start = case["start"]
    graph = BASE_WORLD_MODEL["navigation_graph"]
    map_nodes = {}
    for loc in graph:
        map_nodes[loc] = {
            "temp_have": BASE_WORLD_MODEL["current_room_state"]["inventory"] if loc == start else [],
            "have": ROOM_OBJECTS.get(loc, []),
            "may_have": [],
            "direction": graph.get(loc, {}),
            "may_direction": BASE_WORLD_MODEL["frontier"].get(loc, []),
        }
        local = ROOM_LOCAL_STATE.get(loc)
        if local and local.get("object_states"):
            map_nodes[loc]["relations"] = [
                {"subject": name, "relation": "state", "object": state}
                for name, state in local["object_states"].items()
            ]
        if local and local.get("blocked_exits"):
            map_nodes[loc]["needs"] = [
                f"{direction} blocked: {reason}"
                for direction, reason in local["blocked_exits"].items()
            ]
    return {"current_location": start, "visited_rooms": list(graph.keys()), "map": map_nodes}


## Next-Step Test Cases

Each case asks only for the immediate next command. The expected answer is computed with BFS, but the model is not asked for the full route.


In [ ]:
TEST_CASES = [
    {"id": "room_living_to_troll", "start": "Living Room", "target_location": "Troll Room", "question": "You are in Living Room and want to reach Troll Room. What should your next navigation command be?"},
    {"id": "room_loud_to_kitchen", "start": "Loud Room", "target_location": "Kitchen", "question": "You are in Loud Room and want to return to Kitchen. What should your next navigation command be?"},
    {"id": "room_clearing_to_living", "start": "Clearing", "target_location": "Living Room", "question": "You are in Clearing and want to reach Living Room. What should your next navigation command be?"},
    {"id": "room_kitchen_to_troll", "start": "Kitchen", "target_location": "Troll Room", "question": "You are in Kitchen and want to reach Troll Room. What should your next navigation command be?"},
    {"id": "room_troll_to_kitchen", "start": "Troll Room", "target_location": "Kitchen", "question": "You are in Troll Room and want to reach Kitchen. What should your next navigation command be?"},
    {"id": "room_ewp_to_kitchen", "start": "East-West Passage", "target_location": "Kitchen", "question": "You are in East-West Passage and want to reach Kitchen. What should your next navigation command be?"},
    {"id": "room_north_house_to_tree", "start": "North of House", "target_location": "Up a Tree", "question": "You are at North of House and want to get to Up a Tree. What should your next navigation command be?"},
    {"id": "room_living_to_tree", "start": "Living Room", "target_location": "Up a Tree", "question": "You are in Living Room and want to reach Up a Tree. What should your next navigation command be?"},
    {"id": "room_behind_to_living", "start": "Behind House", "target_location": "Living Room", "question": "You are Behind House and want to reach Living Room. What should your next navigation command be?"},
    {"id": "room_attic_to_troll", "start": "Attic", "target_location": "Troll Room", "question": "You are in Attic and want to reach Troll Room. What should your next navigation command be?"},
    {"id": "room_cellar_to_tree", "start": "Cellar", "target_location": "Up a Tree", "question": "You are in Cellar and want to reach Up a Tree. What should your next navigation command be?"},
    {"id": "room_west_house_to_troll", "start": "West of House", "target_location": "Troll Room", "question": "You are at West of House and want to reach Troll Room. What should your next navigation command be?"},
    {"id": "room_loud_to_round", "start": "Loud Room", "target_location": "Round Room", "question": "You are in Loud Room and want to reach Round Room. What should your next navigation command be?"},
    {"id": "room_kitchen_to_clearing", "start": "Kitchen", "target_location": "Clearing", "question": "You are in Kitchen and want to reach Clearing. What should your next navigation command be?"},
    {"id": "room_round_to_living", "start": "Round Room", "target_location": "Living Room", "question": "You are in Round Room and want to reach Living Room. What should your next navigation command be?"},
    {"id": "object_living_to_egg", "start": "Living Room", "target_object": "egg", "question": "You are in Living Room and need the egg. It may be several rooms away. What should your next navigation command be?"},
    {"id": "object_kitchen_to_axe", "start": "Kitchen", "target_object": "axe", "question": "You are in Kitchen and need the axe. What should your next navigation command be?"},
    {"id": "object_loud_to_bottle", "start": "Loud Room", "target_object": "bottle", "question": "You are in Loud Room and need the bottle. What should your next navigation command be?"},
    {"id": "object_clearing_to_lantern", "start": "Clearing", "target_object": "lantern", "question": "You are in Clearing and need the lantern. What should your next navigation command be?"},
    {"id": "object_troll_to_sack", "start": "Troll Room", "target_object": "sack", "question": "You are in Troll Room and need the sack. What should your next navigation command be?"},
    {"id": "unknown_room_living_to_dam", "start": "Living Room", "target_location": "Dam", "question": "You are in Living Room and want to reach Dam. What should your next navigation command be?"},
    {"id": "unknown_object_living_to_rope", "start": "Living Room", "target_object": "rope", "question": "You are in Living Room and need the rope. What should your next navigation command be?"},
]

def expected_for_case(case):
    graph = BASE_WORLD_MODEL["navigation_graph"]
    start = case["start"]
    target_object = case.get("target_object")
    target_location = case.get("target_location")
    if target_object:
        target_location = BASE_WORLD_MODEL["known_object_locations"].get(target_object)
        if not target_location:
            return {"known": False, "target_object": target_object, "target_location": None, "next_command": None, "next_room": None, "route": [], "path": []}
    route = shortest_route(graph, start, target_location)
    if not route or not route["route"]:
        return {"known": False, "target_object": target_object, "target_location": target_location, "next_command": None, "next_room": None, "route": [], "path": []}
    return {
        "known": True,
        "target_object": target_object,
        "target_location": target_location,
        "next_command": route["route"][0],
        "next_room": route["path"][1],
        "route": route["route"],
        "path": route["path"],
    }

print("Expected next-step answers:")
for case in TEST_CASES:
    expected = expected_for_case(case)
    print("\n" + case["id"])
    print("Q:", case["question"])
    print("known:", expected["known"])
    print("target_location:", expected.get("target_location"))
    if expected.get("target_object"):
        print("target_object:", expected.get("target_object"))
    print("next_command:", expected.get("next_command"))
    print("next_room:", expected.get("next_room"))
    print("full_route_for_reference:", expected.get("route"))


## Run Probe

The grader accepts only the exact immediate next direction for known targets. It does not require the full route.


In [ ]:
SYSTEM_PROMPT = """You are a next-step navigation evaluator for a text adventure agent.
Use ONLY the provided JSON world model. Do not use outside knowledge about Zork.
Your task is to choose the immediate next navigation command, not a full route.
Return JSON only between |start| and |end|.

Required output schema:
{
  "known": true or false,
  "start": "room name",
  "target_location": "room name or null",
  "target_object": "object name or null",
  "next_command": "single direction command or null",
  "next_room": "room reached by next_command or null",
  "reason": "one short sentence"
}

Rules:
- next_command must be exactly one navigation direction, such as "north", "south", "east", "west", "up", or "down".
- Do not output a full route in next_command.
- If target_next_step_hint exists, prefer that next_command exactly.
- If navigation_graph exists, confirmed exits are in navigation_graph[room].
- If navigation_edges exists, each edge string has the form "From Room --direction--> To Room".
- If map exists, this is the current KG format: confirmed exits are in map[room]["direction"], room objects are in map[room]["have"], inventory is in map[current_location]["temp_have"], and may_direction is only unconfirmed possible exits.
- For object questions, use known_object_locations if present; otherwise infer the object's room from map[room]["have"].
- If the target room/object is unknown or no connected route is known, set known=false, next_command=null, next_room=null.
"""

TEST_MODES = [
    {"id": "current_kg_format", "format": "kg", "include_next_step_hint": False},
    {"id": "raw_graph", "format": "clean", "include_next_step_hint": False},
    {"id": "json_edge_list", "format": "edge", "include_next_step_hint": False},
    {"id": "assisted_next_step", "format": "clean", "include_next_step_hint": True},
]

def build_world_for_mode(case, mode):
    if mode["format"] == "kg":
        return build_current_kg_world(case)
    if mode["format"] == "edge":
        return build_edge_world(case)
    return build_clean_world(case, include_next_step_hint=mode["include_next_step_hint"])

def build_user_prompt(case, world):
    return f"""World model JSON:
{json.dumps(world, indent=2)}

Question:
{case['question']}
"""

def ask_qwen(case, world):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(case, world)},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = output[0][inputs.input_ids.shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

def parse_answer(raw):
    m = re.search(r"\|start\|\s*(.*?)\s*\|end\|", raw, re.S)
    body = m.group(1).strip() if m else raw.strip()
    try:
        return json.loads(body)
    except Exception as e:
        return {"parse_error": str(e), "raw_body": body}

def normalize_command(command):
    if command is None:
        return None
    text = str(command).strip().lower()
    return text or None

def answer_matches(answer, expected):
    if "parse_error" in answer:
        return False
    if bool(answer.get("known")) != bool(expected.get("known")):
        return False
    if not expected.get("known"):
        return normalize_command(answer.get("next_command")) is None
    return normalize_command(answer.get("next_command")) == expected.get("next_command")

ALL_RESULTS = []
for mode in TEST_MODES:
    mode_results = []
    print("\n" + "#" * 100)
    print("MODE:", mode["id"], "| include_next_step_hint=", mode["include_next_step_hint"])
    for case in TEST_CASES:
        world = build_world_for_mode(case, mode)
        expected = expected_for_case(case)
        print("\n" + "=" * 90)
        print(case["id"])
        print(case["question"])
        if mode["format"] == "kg":
            print("CURRENT KG ROOM:", json.dumps(world["map"].get(case["start"], {}), indent=2))
        elif mode["format"] == "edge":
            print("CURRENT ROOM STATE:", json.dumps(world["current_room_state"], indent=2))
            print("EDGE COUNT:", len(world["navigation_edges"]))
        elif mode["include_next_step_hint"]:
            print("NEXT STEP HINT:", world.get("target_next_step_hint"))
        print("EXPECTED:", expected)
        raw = ask_qwen(case, world)
        parsed = parse_answer(raw)
        ok = answer_matches(parsed, expected)
        print("RAW:\n", raw)
        print("PARSED:", parsed)
        print("PASS:", ok)
        row = {"mode": mode["id"], "case": case, "expected": expected, "raw": raw, "parsed": parsed, "pass": ok}
        mode_results.append(row)
        ALL_RESULTS.append(row)
    print("\nSUMMARY FOR", mode["id"])
    print(sum(1 for r in mode_results if r["pass"]), "/", len(mode_results), "passed")

print("\nCOMPARISON")
for mode in TEST_MODES:
    rows = [r for r in ALL_RESULTS if r["mode"] == mode["id"]]
    print(mode["id"], sum(1 for r in rows if r["pass"]), "/", len(rows))
